# 01 — Descriptors, Properties, and Metaclasses

Learn attribute lookup, reusable validating fields, method binding, and the limited use cases for dynamic class construction.

*Advanced · 90–150 min*

## Objectives
- Explain how properties implement descriptors
- Write __get__, __set__ and __set_name__ methods
- Understand bound methods
- Evaluate trade-offs of metaclasses vs ordinary classes

## Examples

### 1. Properties and derived state

A property can validate assignments or expose a calculated value without storing redundant fields. Avoid hiding costly I/O inside a property, where readers may expect a cheap attribute lookup.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius
    @property
    def celsius(self):
        return self._celsius
    @celsius.setter
    def celsius(self, value):
        self._celsius = float(value)
    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

value = Temperature(20)
value.celsius = 25
print(value.fahrenheit)

**Design question:** Would a derived fahrenheit field risk becoming stale if stored directly?

### 2. Data descriptors for reusable validation

Descriptors belong to the class, but their per-instance data should be stored on each instance. __set_name__ gives the descriptor its attribute name when the owning class is created. Avoid shared state held on a descriptor.

In [ ]:
class Positive:
    def __set_name__(self, owner, name):
        self.name = name
        self.private = "_" + name
    def __get__(self, obj, owner):
        return self if obj is None else obj.__dict__[self.private]
    def __set__(self, obj, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)) or value <= 0:
            raise ValueError(f"{self.name} must be positive")
        obj.__dict__[self.private] = value

class Product:
    price = Positive()
    def __init__(self, price):
        self.price = price

first, second = Product(5), Product(9)
first.price = 7
print(first.price, second.price)

**Design question:** Why would assigning to self.value inside Positive.__set__ be wrong?

### 3. Method binding

When a function is accessed as an instance method, its descriptor machinery binds self. Access through the class leaves the instance argument explicit. This explains method signatures and monkey-patching behavior.

In [ ]:
class Greeter:
    def hello(self, name):
        return f"Hello, {name}"

greeter = Greeter()
print(greeter.hello("Ada"))
print(Greeter.hello(greeter, "Ada"))
print(greeter.hello.__self__ is greeter)

**Design question:** What is the difference between a bound method and the underlying function?

### 4. Dynamic classes and restraint

type(name, bases, namespace) creates a class dynamically. This can help frameworks but should not replace straightforward declarations. Prefer class decorators, factories, or a normal class when possible.

In [ ]:
def describe(self):
    return f"Plugin: {self.name}"

Plugin = type("Plugin", (), {"describe": describe})
instance = Plugin()
instance.name = "serializer"
print(instance.describe(), isinstance(instance, Plugin))

**Design question:** What complexity does this dynamic example introduce compared with class Plugin: ...?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/01_internals_typing_architecture_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/01', {1: '9b967fa72458772c', 2: '301976d13316e842', 3: 'fab4e52a6957a16b', 4: '5924a39c234ca9cf'})

**Exercise 1.** A read-only computed attribute with `property`.

In [ ]:
class Circle:
    def __init__(self, r):
        self.r = r
    @property
    def diameter(self):
        return ___

c = Circle(3)
try:
    c.diameter = 10
    writable = True
except AttributeError:
    writable = False
check(1, (c.diameter, writable))

**Exercise 2.** A descriptor that stores values under the attribute's own name.

In [ ]:
class Positive:
    def __set_name__(self, owner, name):
        self.name = ___
    def __get__(self, obj, objtype=None):
        return obj.__dict__[self.name]
    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self.name} must be positive")
        obj.__dict__[self.name] = value

class Order:
    qty = Positive()
    def __init__(self, qty):
        self.qty = qty

try:
    Order(0)
    error = None
except ValueError as e:
    error = str(e)
check(2, (Order(5).qty, error))

**Exercise 3.** Bound methods remember their object. Predict the results.

In [ ]:
class Greeter:
    def __init__(self, name): self.name = name
    def hello(self): return f"hi {self.name}"

say = Greeter("ana").hello          # no call yet
predictions = {"say()": ___, "say.__self__.name": ___}
check(3, predictions)

**Exercise 4.** Prefer `__init_subclass__` to a metaclass for registering subclasses.

In [ ]:
class Plugin:
    registry = {}
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Plugin.registry[cls.__name__.lower()] = ___

class Csv(Plugin): pass
class Json(Plugin): pass
check(4, sorted((name, plugin.__name__) for name, plugin in Plugin.registry.items()))

## Common mistakes

- Store per-object data on instances, not on shared descriptors.
- Class construction magic rarely belongs in ordinary application code.
- Properties should not hide unexpected network or disk activity.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** A @property is implemented as a…

   - **a)** Metaclass
   - **b)** Descriptor
   - **c)** Decorator that copies the method

**2.** __set_name__ is called…

   - **a)** When an instance is deleted
   - **b)** When the owner class is created
   - **c)** On every attribute access

**3.** Before writing a metaclass, prefer…

   - **a)** Global variables
   - **b)** Decorators, __init_subclass__, or plain functions
   - **c)** Two metaclasses

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/01', {1: 'c6edf5295ed5', 2: '591fe465f1c7', 3: '61662b4b3d92'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [02 — Object References, Copying, Garbage Collection, and Weakrefs](02_memory_references_weakrefs.ipynb)